# Day 195 — Month 12 Capstone Part 4: Deployment
**Program:** 12-Month Data Science & AI Curriculum | Month 12 — Netherlands Capstone
**Project:** TeleServe India — Customer Complaint Resolution Intelligence
**Repo:** `Month12-Capstone-Netherlands-Portfolio` (branch: `master`)

**Where we are:**
- Day192: Architecture defined (5 layers) — PERFECT
- Day193: Layer 1–2 Data pipeline (`TeleServeCleaningPipeline`) — PERFECT, zero resubmit
- Day194: Layer 3 (TF-IDF + LogisticRegression) + Layer 4 (FAISS + LangChain/Groq RAG) — PERFECT after 1 resubmit
- **Day195 (today): Layer 5 — Deployment.** Build the Streamlit app that *loads* Day194's persisted artifacts (no retraining), with the **RAG resolution-suggester as the primary feature** and the **ML classifier surfaced only as a supplementary, flagged metric** — this ordering was explicitly locked in Day194's Action bullet.

**Rule reminder (NRA):** Number from printed output only. Reason = mechanism, stop there. Action = named owner + timeline, zero hedging.


## Section 1 — Raw Data (DO NOT MODIFY)
Regenerated with `seed=192` so this notebook runs standalone. This is identical in generation logic to Day192 — it exists here only so Layer 1–2 can be re-run to feed Layer 3/4 if you are starting a fresh runtime without your persisted `.joblib`/`.index` files.

In [1]:
# RAW DATA CELL — DO NOT MODIFY ANY LOGIC BELOW THIS LINE
import numpy as np
import pandas as pd
import random

SEED = 192
random.seed(SEED)
np.random.seed(SEED)

n_base = 400
plan_types_clean = ['Prepaid', 'Postpaid', 'Broadband']
casing_variants = ['prepaid', 'PREPAID', 'Postpaid ', 'postpaid', 'BROADBAND', 'Broadband']

rows = []
for i in range(n_base):
    plan = random.choice(plan_types_clean)
    tenure = random.randint(1, 60)
    bill = round(random.uniform(199, 1999), 2)
    complaint = random.choice([
        "network down in my area", "billing amount incorrect", "slow internet speed",
        "sim not activated", "customer care not responding", "recharge failed but amount deducted",
        "data pack not applied", "call drop issue", "wrong plan activated", "refund not received"
    ])
    resolved = random.random() < 0.72          # <-- ground-truth generator, confirmed independent of features
    days_to_resolve = random.randint(1, 10) if resolved else None
    satisfaction_score = random.randint(1, 5) if resolved else None  # unresolved -> missing (MNAR, not MCAR)
    rows.append([i, plan, tenure, bill, complaint, resolved, days_to_resolve, satisfaction_score])

df_raw = pd.DataFrame(rows, columns=[
    'customer_id', 'plan_type', 'tenure_months', 'monthly_bill_inr',
    'complaint_text', 'resolved', 'days_to_resolve', 'satisfaction_score'
])

# Inject casing inconsistency (multi-channel manual entry simulation)
casing_idx = np.random.choice(df_raw.index, 6, replace=False)
for idx, cv in zip(casing_idx, casing_variants):
    df_raw.loc[idx, 'plan_type'] = cv

# Inject 9 duplicate rows
dup_rows = df_raw.sample(9, random_state=SEED)
df_raw = pd.concat([df_raw, dup_rows], ignore_index=True)

print(f"Raw shape: {df_raw.shape}")
print(f"Missing satisfaction_score: {df_raw['satisfaction_score'].isna().sum()}")
print(f"Duplicate rows (exact): {df_raw.duplicated().sum()}")


Raw shape: (409, 8)
Missing satisfaction_score: 103
Duplicate rows (exact): 9


## Section 2 — Concept Notes: Deployment (Layer 5)

**What it is:** Deployment is the layer that turns trained/persisted artifacts (Layer 3 model, Layer 4 FAISS index + KB) into something a user or client can actually interact with — here, a Streamlit app. Deployment code **never retrains**; it **loads**.

**Why it matters (client work):** A client does not care that your model exists in a notebook — they care that they can open a URL, type a complaint, and get an answer. The gap between "notebook works" and "app works" is where most freelance DS deliverables lose credibility.

**Key deployment principles:**
1. **Load, don't retrain.** `joblib.load()` / `faiss.read_index()` at app startup. Retraining inside a deployed app is a red flag in any review — it means the app is non-deterministic and slow.
2. **Primary vs. supplementary feature ordering is a product decision, not a technical afterthought.** Day194 showed the ML classifier (0.730 acc) never beat the majority-class baseline (0.730) — meaning `resolved` is not predictable from available features. Shipping it as the *primary* feature would overstate what the model can do. The RAG retrieval (grounded in real past precedents) has no such accuracy claim to inflate — it just surfaces similar cases. That is why RAG is primary and ML is a flagged, secondary readout.
3. **Test gate before "deployed."** A deployment is not done when `streamlit run` doesn't crash — it's done when a fixed test CSV (matching exact column names, category values, and level values from training data — per project rule #12 on Streamlit test CSVs) runs through the app logic and produces expected shape/values.
4. **Flagging, not hiding.** Per Day194's Action, the ML output is shown with a visible caveat: "This does not outperform guessing the majority class — treat as directional only." This is an ethical labeling decision, not a UI nicety: an unlabeled 73%-accuracy number reads as confident to a non-technical client when it is not.

**Real-world use:** This exact pattern (retrieval-grounded suggestion as primary, weak classifier flagged as secondary signal) is standard when a supervised model's features don't causally explain the label — common in support/ops data where the labeled outcome depends on unobserved agent/process factors, not on the fields you have.


## Section 3 — Practice Tasks (complete independently)

Complete Tasks 1–5 in the cells below. Comment skeletons show WHAT to do — you write the HOW.
Do not look at Section 4 (Answer Key) until you have attempted all 5 tasks.


In [2]:
# TASK 1 — Rebuild Layer 1-2: Cleaning Pipeline (from Day193)
# Goal: reproduce df_clean from df_raw so Layer 3/4 have input if starting fresh.
# Method: implement TeleServeCleaningPipeline class with one operation per method,
#         run operations in correct order (normalize → dedupe → verify), and gate with tests.

class TeleServeCleaningPipeline:
    def __init__(self, raw_df):
        self.raw_df = raw_df.copy()
        self.df = None

    def load(self):
        """Copy raw data into self.df without mutating original."""
        self.df = self.raw_df.copy()

    def normalize_plan_type(self):
        """Strip whitespace and capitalize plan_type to canonical values."""
        self.df["plan_type"] = self.df["plan_type"].str.strip().str.capitalize()

    def drop_duplicates(self):
        """Remove exact duplicate rows after casing normalization."""
        self.df = self.df.drop_duplicates().reset_index(drop=True)

    def verify_satisfaction_untouched(self):
        """Ensure MNAR preservation: no imputation on unresolved rows."""
        mismatched = self.df[(self.df["resolved"] == False) & (self.df["satisfaction_score"].notna())]
        if len(mismatched) > 0:
            raise AssertionError("satisfaction_score was imputed where it should stay NaN")

    def run_tests(self):
        """Assertion gate – prints PASS/FAIL for each check, raises on any failure."""
        checks = {
            "plan_type has 3 unique values": self.df["plan_type"].nunique() == 3,
            "No duplicate rows": self.df.duplicated().sum() == 0,
            "Total rows == 400 (409 raw - 9 duplicates)": len(self.df) == 400,
            "No imputation on unresolved (resolved==False)":
                not ((self.df["resolved"] == False) & (self.df["satisfaction_score"].notna())).any(),
            "All resolved have satisfaction score":
                not ((self.df["resolved"] == True) & (self.df["satisfaction_score"].isna())).any(),
        }
        all_pass = True
        for desc, cond in checks.items():
            if cond:
                print(f"PASS: {desc}")
            else:
                print(f"FAIL: {desc}")
                all_pass = False
        if not all_pass:
            raise AssertionError("Test suite failed – cleaning output invalid")

    def run(self):
        """Orchestrate all steps and return cleaned DataFrame."""
        self.load()
        self.normalize_plan_type()
        self.drop_duplicates()
        self.verify_satisfaction_untouched()
        self.run_tests()
        return self.df

df_clean = TeleServeCleaningPipeline(df_raw).run()
print(f"df_clean shape: {df_clean.shape}")  # Number for this task
# Output:
# PASS: plan_type has 3 unique values
# PASS: No duplicate rows
# PASS: Total rows == 400 (409 raw - 9 duplicates)
# PASS: No imputation on unresolved (resolved==False)
# PASS: All resolved have satisfaction score
# df_clean shape: (400, 8)

PASS: plan_type has 3 unique values
PASS: No duplicate rows
PASS: Total rows == 400 (409 raw - 9 duplicates)
PASS: No imputation on unresolved (resolved==False)
PASS: All resolved have satisfaction score
df_clean shape: (400, 8)


In [3]:
# TASK 2 — Rebuild/Persist Layer 3 (ML) + Layer 4 (RAG) artifacts
# Goal: recreate Day194's artifacts EXACTLY so app.py in Task 4 has something real to load.
# Method: train TF-IDF + LogisticRegression, build FAISS index over resolved complaints,
#         persist all objects, and save baseline accuracy to a file.

import subprocess
import sys
import joblib
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# --- Ensure faiss is installed ---
try:
    import faiss
except ImportError:
    print("Installing faiss-cpu...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "faiss-cpu"])
    import faiss

# --- Layer 3: ML ---
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
X_text = tfidf.fit_transform(df_clean['complaint_text'])

plan_dummies = pd.get_dummies(df_clean['plan_type'], prefix='plan')
X_numeric = pd.concat([df_clean[['tenure_months', 'monthly_bill_inr']], plan_dummies], axis=1)
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

X_full = hstack([X_text, csr_matrix(X_numeric_scaled)])
y = df_clean['resolved'].astype(int)

# Train-test split (same as Day194)
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.25, random_state=192, stratify=y
)

clf = LogisticRegression(max_iter=1000, random_state=192)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
baseline = max(y_test.mean(), 1 - y_test.mean())

print(f"ML accuracy: {accuracy:.3f}, Baseline accuracy: {baseline:.3f}")  # Number for Task 3

# Save baseline for the app to read dynamically
with open("baseline_accuracy.txt", "w") as f:
    f.write(f"{baseline:.3f}")

joblib.dump(tfidf, "layer3_tfidf.joblib")
joblib.dump(scaler, "layer3_scaler.joblib")
joblib.dump(clf, "layer3_model.joblib")

# --- Layer 4: RAG ---
resolved_df = df_clean[df_clean["resolved"] == True].reset_index(drop=True)
print(f"Resolved KB size: {len(resolved_df)}")

kb_tfidf = TfidfVectorizer(max_features=200, stop_words="english")
kb_vectors = kb_tfidf.fit_transform(resolved_df["complaint_text"]).toarray().astype("float32")
faiss.normalize_L2(kb_vectors)

index = faiss.IndexFlatIP(kb_vectors.shape[1])
index.add(kb_vectors)

def retrieve_similar(query, k=3):
    query_vec = kb_tfidf.transform([query]).toarray().astype("float32")
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(resolved_df):
            row = resolved_df.iloc[idx]
            results.append({
                "complaint_text": row["complaint_text"],
                "days_to_resolve": row["days_to_resolve"],
                "satisfaction_score": row["satisfaction_score"],
                "similarity": float(score)
            })
    return results

faiss.write_index(index, "layer4_faiss.index")
joblib.dump(kb_tfidf, "layer4_kb_tfidf.joblib")
resolved_df.to_csv("layer4_resolved_kb.csv", index=False)

print("All artifacts saved.")
# Output:
# Installing faiss-cpu...
# ML accuracy: 0.750, Baseline accuracy: 0.750
# Resolved KB size: 300
# All artifacts saved.

ML accuracy: 0.750, Baseline accuracy: 0.750
Resolved KB size: 300
All artifacts saved.


In [6]:
# TASK 3 — NRA Business Case: Deployment Architecture Decision (code cell)
# Number: from live variables accuracy and baseline (Task 2)
# Reason: causal mechanism only
# Action: named owner + threshold + review date

nra_deployment = {
    "number": f"ML accuracy = {accuracy:.3f}, baseline = {baseline:.3f}",
    "reason": "The ML model's accuracy equals the majority-class baseline, meaning the label 'resolved' has no learnable relationship with any of the available features (tenure, bill, plan, complaint text).",
    "action": "Deploy RAG retrieval as the primary feature. The ML classifier will be shown only as a supplementary, visually de‑emphasized metric with a caption 'Directional only – does not outperform baseline'. Re‑evaluation threshold: if new features become available, the model must beat the baseline by at least 0.05 on a validation set; this will be assessed by the data science lead on the next quarterly review (2026‑10‑01)."
}

print(f"Number: {nra_deployment['number']}")
print(f"Reason: {nra_deployment['reason']}")
print(f"Action: {nra_deployment['action']}")

Number: ML accuracy = 0.750, baseline = 0.750
Reason: The ML model's accuracy equals the majority-class baseline, meaning the label 'resolved' has no learnable relationship with any of the available features (tenure, bill, plan, complaint text).
Action: Deploy RAG retrieval as the primary feature. The ML classifier will be shown only as a supplementary, visually de‑emphasized metric with a caption 'Directional only – does not outperform baseline'. Re‑evaluation threshold: if new features become available, the model must beat the baseline by at least 0.05 on a validation set; this will be assessed by the data science lead on the next quarterly review (2026‑10‑01).


In [7]:
# TASK 4 — Build app.py (Streamlit deployment file) – FIXED COLUMN NAMES
%%writefile app.py
import streamlit as st
import joblib
import faiss
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix

@st.cache_resource
def load_artifacts():
    tfidf = joblib.load("layer3_tfidf.joblib")
    scaler = joblib.load("layer3_scaler.joblib")
    model = joblib.load("layer3_model.joblib")
    index = faiss.read_index("layer4_faiss.index")
    kb_tfidf = joblib.load("layer4_kb_tfidf.joblib")
    resolved_df = pd.read_csv("layer4_resolved_kb.csv")
    with open("baseline_accuracy.txt", "r") as f:
        baseline = float(f.read())
    return tfidf, scaler, model, index, kb_tfidf, resolved_df, baseline

tfidf, scaler, model, index, kb_tfidf, resolved_df, baseline = load_artifacts()

def ml_predict(complaint_text, tenure, bill, plan):
    X_text = tfidf.transform([complaint_text])
    plan_dummies = pd.get_dummies(pd.Series([plan]), prefix='plan')
    for col in ['plan_Broadband', 'plan_Postpaid', 'plan_Prepaid']:
        if col not in plan_dummies.columns:
            plan_dummies[col] = 0
    plan_dummies = plan_dummies[['plan_Broadband', 'plan_Postpaid', 'plan_Prepaid']]

    # Build feature vector in the EXACT order and column names as training
    full_numeric = np.hstack([[tenure, bill], plan_dummies.values[0]])
    full_numeric_df = pd.DataFrame(
        full_numeric.reshape(1, -1),
        columns=['tenure_months', 'monthly_bill_inr', 'plan_Broadband', 'plan_Postpaid', 'plan_Prepaid']
    )
    full_numeric_scaled = scaler.transform(full_numeric_df)
    X_numeric_sparse = csr_matrix(full_numeric_scaled)
    X_full = hstack([X_text, X_numeric_sparse])
    pred = model.predict(X_full)[0]
    prob = model.predict_proba(X_full)[0][1]
    return pred, prob

def retrieve_similar(query, k=3):
    query_vec = kb_tfidf.transform([query]).toarray().astype("float32")
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(resolved_df):
            row = resolved_df.iloc[idx]
            results.append({
                "complaint_text": row["complaint_text"],
                "days_to_resolve": row["days_to_resolve"],
                "satisfaction_score": row["satisfaction_score"],
                "similarity": float(score)
            })
    return results

st.set_page_config(page_title="TeleServe Resolution Assistant", layout="wide")
st.title("📞 TeleServe India – Complaint Resolution Intelligence")
st.markdown("Enter a customer complaint below to get similar resolved cases and an ML sanity check.")

with st.form("complaint_form"):
    complaint = st.text_area("Complaint text", height=100)
    tenure = st.number_input("Tenure (months)", min_value=0, max_value=120, value=12)
    bill = st.number_input("Monthly bill (INR)", min_value=0, max_value=5000, value=500)
    plan = st.selectbox("Plan type", ["Prepaid", "Postpaid", "Broadband"])
    submitted = st.form_submit_button("Analyze")

if submitted and complaint:
    st.subheader("🔍 Similar Resolved Complaints (Precedents)")
    similar = retrieve_similar(complaint, k=3)
    if similar:
        for i, row in enumerate(similar, 1):
            with st.expander(f"Case {i} (similarity: {row['similarity']:.3f})"):
                st.write(f"**Complaint:** {row['complaint_text']}")
                st.write(f"**Days to resolve:** {row['days_to_resolve']}")
                st.write(f"**Satisfaction score:** {row['satisfaction_score']}")
    else:
        st.warning("No similar resolved complaints found.")

    with st.expander("⚙️ ML Classifier (directional only)"):
        pred, prob = ml_predict(complaint, tenure, bill, plan)
        st.caption(f"This model does not outperform the majority‑class baseline ({baseline:.3f} accuracy). Treat as directional only.")
        if pred == 1:
            st.info(f"Prediction: **Resolved** (confidence: {prob:.2f})")
        else:
            st.warning(f"Prediction: **Not Resolved** (confidence: {1-prob:.2f})")
else:
    st.info("Please fill in the complaint text and click Analyze.")

Overwriting app.py


In [8]:
# TASK 5 — Test Gate (FIXED COLUMN NAMES)
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix

test_data = pd.DataFrame({
    "complaint_text": [
        "network down in my area",
        "billing amount incorrect",
        "slow internet speed",
        "sim not activated",
        "customer care not responding",
        "recharge failed but amount deducted",
        "data pack not applied",
        "call drop issue",
    ],
    "tenure_months": [12, 24, 6, 36, 48, 18, 9, 30],
    "monthly_bill_inr": [500, 800, 300, 1200, 650, 400, 700, 950],
    "plan_type": ["Postpaid", "Prepaid", "Broadband", "Postpaid", "Prepaid", "Broadband", "Prepaid", "Postpaid"]
})
test_csv_path = "test_input.csv"
test_data.to_csv(test_csv_path, index=False)

def run_smoke_test():
    import joblib
    import faiss
    tfidf = joblib.load("layer3_tfidf.joblib")
    scaler = joblib.load("layer3_scaler.joblib")
    model = joblib.load("layer3_model.joblib")
    index = faiss.read_index("layer4_faiss.index")
    kb_tfidf = joblib.load("layer4_kb_tfidf.joblib")
    resolved_df = pd.read_csv("layer4_resolved_kb.csv")

    def ml_predict(complaint_text, tenure, bill, plan):
        X_text = tfidf.transform([complaint_text])
        plan_dummies = pd.get_dummies(pd.Series([plan]), prefix='plan')
        for col in ['plan_Broadband', 'plan_Postpaid', 'plan_Prepaid']:
            if col not in plan_dummies.columns:
                plan_dummies[col] = 0
        plan_dummies = plan_dummies[['plan_Broadband', 'plan_Postpaid', 'plan_Prepaid']]
        full_numeric = np.hstack([[tenure, bill], plan_dummies.values[0]])
        full_numeric_df = pd.DataFrame(
            full_numeric.reshape(1, -1),
            columns=['tenure_months', 'monthly_bill_inr', 'plan_Broadband', 'plan_Postpaid', 'plan_Prepaid']
        )
        full_numeric_scaled = scaler.transform(full_numeric_df)
        X_numeric_sparse = csr_matrix(full_numeric_scaled)
        X_full = hstack([X_text, X_numeric_sparse])
        pred = model.predict(X_full)[0]
        return pred

    def retrieve_similar(query, k=3):
        query_vec = kb_tfidf.transform([query]).toarray().astype("float32")
        faiss.normalize_L2(query_vec)
        scores, indices = index.search(query_vec, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(resolved_df):
                row = resolved_df.iloc[idx]
                results.append({
                    "complaint_text": row["complaint_text"],
                    "days_to_resolve": row["days_to_resolve"],
                    "satisfaction_score": row["satisfaction_score"],
                    "similarity": float(score)
                })
        return results

    test_df = pd.read_csv(test_csv_path)
    all_pass = True
    for idx, row in test_df.iterrows():
        try:
            pred = ml_predict(row["complaint_text"], row["tenure_months"], row["monthly_bill_inr"], row["plan_type"])
            assert pred in [0, 1], "Prediction not 0/1"
            retrieved = retrieve_similar(row["complaint_text"], k=3)
            assert len(retrieved) == 3, f"Expected 3 retrieved, got {len(retrieved)}"
            print(f"PASS row {idx}: prediction={pred}, retrieved={len(retrieved)}")
        except Exception as e:
            print(f"FAIL row {idx}: {e}")
            all_pass = False

    if all_pass:
        print("\n✅ ALL TESTS PASSED – app.py is ready for deployment.")
    else:
        print("\n❌ SOME TESTS FAILED – fix issues before deploying.")

run_smoke_test()

PASS row 0: prediction=1, retrieved=3
PASS row 1: prediction=1, retrieved=3
PASS row 2: prediction=1, retrieved=3
PASS row 3: prediction=1, retrieved=3
PASS row 4: prediction=1, retrieved=3
PASS row 5: prediction=1, retrieved=3
PASS row 6: prediction=1, retrieved=3
PASS row 7: prediction=1, retrieved=3

✅ ALL TESTS PASSED – app.py is ready for deployment.


## Interview Framing Answer

**"How would you explain to a client why your recommendation engine uses retrieval instead of a prediction model as the primary feature?"**

> *Our prediction model achieved 75% accuracy, which is exactly the same as guessing the majority outcome. That tells us the available data – tenure, bill, plan, and even the complaint text – doesn’t contain enough signal to reliably predict whether a complaint will be resolved. Instead of pretending otherwise, I made the retrieval system the primary feature: it searches through actual resolved complaints with known outcomes and shows the agent the most similar past cases, along with their resolution time and satisfaction score. This is transparent, grounded, and doesn’t overstate what the model can do. The ML classifier is still shown, but clearly flagged as ‘directional only’ so the agent knows not to rely on it.*

## Section 4 — Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 — Cleaning pipeline rebuild | 15 | Correct order (normalize before dedupe), all 4 assertions pass, correct Number printed |
| Task 2 — Layer 3/4 rebuild + persist | 20 | Accuracy vs baseline printed and compared, correct resolved-only KB size, artifacts actually written to disk |
| Task 3 — NRA business case | 20 | Number from live output; Reason = mechanism only (no benefit-language bleed); Action has named owner + explicit re-evaluation threshold |
| Task 4 — app.py deployment file | 25 | Loads artifacts (no retraining), RAG is visually primary, ML is visually secondary + captioned "directional only" |
| Task 5 — Test gate | 15 | Test CSV has exact column names/values from training data; smoke test asserts on real exceptions/shape; prints explicit PASS/FAIL gate |
| Communication (interview framing) | 5 | Clear plain-language explanation of why RAG is primary and ML is flagged |
| **Total** | **100** | |

**Interview-framing prompt (answer in your own words before submitting):**
"How would you explain to a client why your recommendation engine uses retrieval instead of a prediction model as the primary feature?"
